# Copper Forecast Backtest: Three Market Periods

This notebook compares copper-price forecasts during three distinct periods:

- **February 2011 cutoff:** forecasts March through August 2011.
- **March 2020 cutoff:** forecasts April through September 2020.
- **March 2026 cutoff:** forecasts April through September 2026, scoring only months already observed.

Every method receives exactly the same **24 monthly copper prices** available through its cutoff. The comparison includes simple rules, standard statistical methods, combinations of numerical forecasts, and three agent designs: history only, numerical-model results without news, and cutoff-aware news.

The 2011 and 2020 agent results remain in the combined table as requested. They need extra caution: a modern language model may have encountered descriptions of those historical periods during training, even when its web search is restricted to the forecast date.

## Metric guide

All error measures use only target months for which an actual copper price is available.

- **Mean absolute error (MAE):** the average size of each miss in USD per metric ton, ignoring whether the forecast was high or low. Lower is better.
  $$\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^{n}|\hat y_i-y_i|$$
- **Root mean squared error (RMSE):** another average error measure that gives extra weight to large misses. Lower is better.
  $$\operatorname{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat y_i-y_i)^2}$$
- **Symmetric mean absolute percentage error (sMAPE):** expresses error relative to the sizes of both the forecast and actual price. Lower is better.
  $$\operatorname{sMAPE}=\frac{100}{n}\sum_{i=1}^{n}\frac{2|\hat y_i-y_i|}{|y_i|+|\hat y_i|}$$
- **Mean absolute scaled error (MASE):** divides each model's MAE by the MAE from simply repeating the latest price in the same period. Below 1 means the model beat that simple benchmark.
  $$\operatorname{MASE}=\frac{\operatorname{MAE}_{model}}{\operatorname{MAE}_{last\ value}}$$
- **Bias:** the average signed error. Positive means forecasts were too high on average; negative means they were too low.
  $$\operatorname{Bias}=\frac{1}{n}\sum_{i=1}^{n}(\hat y_i-y_i)$$
- **Direction accuracy:** the share of forecasts that correctly predicted whether price would rise or fall from the cutoff price. Higher is better.
- **Continuous ranked probability score (CRPS):** scores the full forecast range, rewarding ranges that are both accurate and appropriately narrow. Lower is better.
- **80% coverage:** the share of actual prices inside the model's 10th-to-90th percentile range. Over many forecasts, a well-sized 80% range should contain about 80% of actual outcomes.
- **Interval width:** the average distance from the 10th to the 90th percentile. Narrower ranges are more useful only when their coverage remains reliable.

CRPS, coverage, and interval width appear only for methods that naturally produce a range of possible outcomes. Simple methods return one number, so this notebook does **not** invent uncertainty ranges for them. With only three periods and at most 18 scored target months, all summary statistics should be read as descriptive evidence rather than a final verdict.

## 1. Configure the notebook environment

The numerical comparison runs by default and needs only the cached FRED file. Each agent has a separate switch because agent runs use external model calls. Fixed random seeds make sampled numerical forecasts repeatable.

In [3]:
import json
import random
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import properscoring as ps
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.holtwinters import ExponentialSmoothing, Holt


ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").exists() and (path / "aieng-forecasting").exists()
)
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env", override=False)

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.features import StaticFrameAdapter
from aieng.forecasting.evaluation import STANDARD_QUANTILES, ForecastingTask
from aieng.forecasting.methods import DartsAutoARIMAPredictor, DartsKalmanForecasterPredictor
from copper_forecasting.agent import (
    build_copper_agent_predictor,
    build_copper_basic_config,
    build_copper_model_panel_config,
    build_copper_model_panel_predictor,
    build_copper_news_config,
)
from copper_forecasting.data import COPPER_FRED_ID, COPPER_SERIES_ID
from copper_forecasting.prophet_baseline import CopperProphetPredictor

In [4]:
@dataclass(frozen=True)
class BacktestConfig:
    input_months: int = 24
    validation_months: int = 6
    horizons: tuple[int, ...] = (1, 2, 3, 4, 5, 6)
    confidence: float = 0.80
    numerical_samples: int = 200
    cache_path: Path = ROOT / "data" / "fred" / f"{COPPER_FRED_ID}.parquet"


CONFIG = BacktestConfig()
SCENARIOS = {
    "2011 commodity boom": pd.Timestamp("2011-02-01"),
    "2020 pandemic shock": pd.Timestamp("2020-03-01"),
    "2026 recent period": pd.Timestamp("2026-03-01"),
}
RUN_NUMERICAL_METHODS = True
RUN_HISTORY_AGENT = True
RUN_MODEL_PANEL_AGENT = True
RUN_NEWS_AGENT = True
EXPORT_RESULTS = False
AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # advanced

random.seed(42)
np.random.seed(42)

print(CONFIG)
print("Numerical methods:", RUN_NUMERICAL_METHODS)
print("Agent switches:", RUN_HISTORY_AGENT, RUN_MODEL_PANEL_AGENT, RUN_NEWS_AGENT)

BacktestConfig(input_months=24, validation_months=6, horizons=(1, 2, 3, 4, 5, 6), confidence=0.8, numerical_samples=200, cache_path=PosixPath('/home/coder/agentic-forecasting/data/fred/PCOPPUSDM.parquet'))
Numerical methods: True
Agent switches: True True True


## 2. Load and validate the copper prices

This is an evaluation notebook, so it requires the real cached FRED series and never substitutes made-up data. The current cache may include corrections published after the original observation date. That means the price history is suitable for comparing methods consistently, but it is not a perfect reconstruction of the exact file a forecaster would have seen years ago.

In [5]:
if not CONFIG.cache_path.exists():
    raise FileNotFoundError(
        f"Missing {CONFIG.cache_path}. Populate the FRED cache before running this evaluation."
    )

required_columns = {"timestamp", "value", "released_at"}
copper = pd.read_parquet(CONFIG.cache_path)
missing_columns = required_columns.difference(copper.columns)
if missing_columns:
    raise ValueError(f"Copper cache is missing required columns: {sorted(missing_columns)}")

copper = copper.loc[:, ["timestamp", "value", "released_at"]].copy()
copper["timestamp"] = pd.to_datetime(copper["timestamp"])
copper["released_at"] = pd.to_datetime(copper["released_at"])
copper["value"] = pd.to_numeric(copper["value"], errors="coerce")
copper = copper.dropna(subset=["timestamp", "value"]).drop_duplicates("timestamp", keep="last")
copper = copper.sort_values("timestamp").set_index("timestamp")
expected_months = pd.date_range(copper.index.min(), copper.index.max(), freq="MS")
missing_months = expected_months.difference(copper.index)
assert missing_months.empty, f"Monthly observations are missing: {missing_months.tolist()}"

scenario_rows = []
scenario_inputs = {}
scenario_services = {}
for scenario, cutoff in SCENARIOS.items():
    input_start = cutoff - pd.DateOffset(months=CONFIG.input_months - 1)
    target_dates = pd.date_range(cutoff + pd.offsets.MonthBegin(1), periods=max(CONFIG.horizons), freq="MS")
    input_frame = copper.loc[input_start:cutoff].copy()
    assert len(input_frame) == CONFIG.input_months
    assert input_frame.index.max() == cutoff
    assert not (input_frame.index > cutoff).any()
    scenario_inputs[scenario] = input_frame

    service_frame = copper.loc[input_start:].reset_index()
    service = DataService()
    service.register(
        COPPER_SERIES_ID,
        StaticFrameAdapter(service_frame),
        SeriesMetadata(
            series_id=COPPER_SERIES_ID,
            description="Global monthly copper price",
            source="FRED PCOPPUSDM cache",
            units="USD per metric ton",
            frequency="MS",
        ),
    )
    scenario_services[scenario] = service
    observed = int(target_dates.isin(copper.index).sum())
    scenario_rows.append(
        {
            "Scenario": scenario,
            "Input start": input_start.date(),
            "Cutoff": cutoff.date(),
            "Target start": target_dates.min().date(),
            "Target end": target_dates.max().date(),
            "Observed targets": observed,
            "Pending targets": len(target_dates) - observed,
        }
    )

scenario_calendar = pd.DataFrame(scenario_rows)
task = ForecastingTask(
    task_id="copper_six_month_path",
    target_series_id=COPPER_SERIES_ID,
    horizons=list(CONFIG.horizons),
    frequency="MS",
    description="Global monthly copper price for each of the next six months, in USD per metric ton.",
)

print(f"Loaded {len(copper):,} monthly prices from {copper.index.min().date()} to {copper.index.max().date()}.")
display(scenario_calendar)

Loaded 415 monthly prices from 1992-01-01 to 2026-07-01.


,Scenario,Input start,Cutoff,Target start,Target end,Observed targets,Pending targets
0,2011 commodity boom,2009-03-01,2011-02-01,2011-03-01,2011-08-01,6,0
1,2020 pandemic shock,2018-04-01,2020-03-01,2020-04-01,2020-09-01,6,0
2,2026 recent period,2024-04-01,2026-03-01,2026-04-01,2026-09-01,4,2


## 3. See the input and target periods

Each panel separates the 24 prices given to the models from the six months they must forecast. A dotted line marks the last month available as input.

In [6]:
COPPER = "#a65d35"
TEAL = "#16827c"
CHARCOAL = "#262626"
PENDING = "#a8adb3"

period_figure = make_subplots(rows=1, cols=3, subplot_titles=list(SCENARIOS), shared_yaxes=False)
for column, (scenario, cutoff) in enumerate(SCENARIOS.items(), start=1):
    input_frame = scenario_inputs[scenario]
    target_dates = pd.date_range(cutoff + pd.offsets.MonthBegin(1), periods=max(CONFIG.horizons), freq="MS")
    observed_targets = copper.reindex(target_dates)["value"]
    period_figure.add_trace(
        go.Scatter(x=input_frame.index, y=input_frame["value"], name="24-month input", line={"color": COPPER, "width": 3}, showlegend=column == 1),
        row=1,
        col=column,
    )
    period_figure.add_trace(
        go.Scatter(x=target_dates, y=observed_targets, name="Target period", line={"color": TEAL, "width": 3}, mode="lines+markers", showlegend=column == 1),
        row=1,
        col=column,
    )
    period_figure.add_vline(x=cutoff.timestamp() * 1000, line_dash="dot", line_color=CHARCOAL, row=1, col=column)

period_figure.update_layout(
    title="The same 24-month input length in three market periods",
    height=430,
    template="plotly_white",
    legend={"orientation": "h", "y": 1.16},
)
period_figure.update_yaxes(title_text="USD per metric ton", row=1, col=1)
period_figure.show()

## 4. Define the forecasting methods

The simple methods make their assumptions easy to inspect. AutoReg estimates how recent prices relate to earlier prices. AutoARIMA automatically selects a standard time-series pattern from the input data. Kalman forecasting updates an estimated hidden price path as new observations arrive. Prophet estimates a flexible trend.

Twenty-four observations are enough to run this comparison, but they leave complex models with substantial uncertainty. ETS therefore uses a trend without a seasonal component; estimating a full yearly seasonal pattern from only two annual cycles would be fragile. Seasonal naive remains as a transparent “same month last year” benchmark.

Method failures are recorded rather than replaced silently. Forecasts at or below zero are also retained and flagged, because hiding them would make a method look better than it was.

In [7]:
POINT_METHODS = {
    "Last value": ("Simple baseline", "last_value"),
    "Historical mean": ("Simple baseline", "historical_mean"),
    "Drift": ("Simple baseline", "drift"),
    "Seasonal naive": ("Simple baseline", "seasonal_naive"),
    "12-month moving average": ("Simple baseline", "moving_average"),
    "ETS trend": ("Statistical model", "ets"),
    "Damped Holt trend": ("Statistical model", "holt_damped"),
    "AutoReg": ("Statistical model", "autoreg"),
}
PROBABILISTIC_FACTORIES = {
    "Prophet": lambda: CopperProphetPredictor(interval_width=CONFIG.confidence, min_history=24),
    "AutoARIMA": lambda: DartsAutoARIMAPredictor(num_samples=CONFIG.numerical_samples),
    "Kalman": lambda: DartsKalmanForecasterPredictor(num_samples=CONFIG.numerical_samples),
}


def point_method_forecast(train: pd.Series, horizon: int, method: str) -> np.ndarray:
    """Fit one point-only method and return a six-month path."""
    if method == "last_value":
        forecast = np.repeat(train.iloc[-1], horizon)
    elif method == "historical_mean":
        forecast = np.repeat(train.mean(), horizon)
    elif method == "drift":
        slope = (train.iloc[-1] - train.iloc[0]) / max(len(train) - 1, 1)
        forecast = train.iloc[-1] + slope * np.arange(1, horizon + 1)
    elif method == "seasonal_naive":
        if len(train) < 12:
            raise ValueError("Seasonal naive needs at least 12 monthly observations.")
        forecast = np.resize(train.iloc[-12:].to_numpy(), horizon)
    elif method == "moving_average":
        forecast = np.repeat(train.iloc[-12:].mean(), horizon)
    elif method == "ets":
        fitted = ExponentialSmoothing(train, trend="add", seasonal=None, initialization_method="estimated").fit()
        forecast = fitted.forecast(horizon)
    elif method == "holt_damped":
        fitted = Holt(train, damped_trend=True, initialization_method="estimated").fit()
        forecast = fitted.forecast(horizon)
    elif method == "autoreg":
        fitted = AutoReg(train, lags=min(6, len(train) // 4), old_names=False, trend="ct").fit()
        forecast = fitted.forecast(horizon)
    else:
        raise ValueError(f"Unknown point method: {method}")
    return np.asarray(forecast, dtype=float)


def past_only_mae(train: pd.Series, method: str) -> float:
    """Check a method on the final six months of the input window only."""
    fitting_values = train.iloc[: -CONFIG.validation_months]
    held_back_values = train.iloc[-CONFIG.validation_months :]
    forecast = point_method_forecast(fitting_values, CONFIG.validation_months, method)
    return float(np.mean(np.abs(forecast - held_back_values.to_numpy())))


def result_row(
    *,
    scenario: str,
    cutoff: pd.Timestamp,
    model: str,
    family: str,
    horizon: int,
    point_forecast: float,
    quantiles: dict[float, float] | None = None,
    rationale: str = "",
) -> dict[str, object]:
    """Create one consistent result row without inventing missing ranges."""
    forecast_date = cutoff + pd.DateOffset(months=horizon)
    actual = float(copper.loc[forecast_date, "value"]) if forecast_date in copper.index else np.nan
    cutoff_price = float(copper.loc[cutoff, "value"])
    point = float(point_forecast)
    error = point - actual if np.isfinite(actual) else np.nan
    denominator = abs(actual) + abs(point) if np.isfinite(actual) else np.nan
    complete_range = quantiles is not None and all(level in quantiles for level in STANDARD_QUANTILES)
    crps = (
        float(ps.crps_ensemble(actual, np.array([quantiles[level] for level in STANDARD_QUANTILES])))
        if np.isfinite(actual) and complete_range
        else np.nan
    )
    q10 = float(quantiles[0.10]) if quantiles is not None and 0.10 in quantiles else np.nan
    q90 = float(quantiles[0.90]) if quantiles is not None and 0.90 in quantiles else np.nan
    direction_correct = (
        bool(np.sign(point - cutoff_price) == np.sign(actual - cutoff_price)) if np.isfinite(actual) else np.nan
    )
    return {
        "scenario": scenario,
        "cutoff": cutoff,
        "model": model,
        "family": family,
        "horizon": horizon,
        "forecast_date": forecast_date,
        "point_forecast": point,
        "q10": q10,
        "q90": q90,
        "quantiles": quantiles,
        "actual": actual,
        "status": "Scored" if np.isfinite(actual) else "Pending",
        "error": error,
        "absolute_error": abs(error) if np.isfinite(error) else np.nan,
        "squared_error": error**2 if np.isfinite(error) else np.nan,
        "absolute_percentage_component": 200 * abs(error) / denominator if denominator else np.nan,
        "direction_correct": direction_correct,
        "crps": crps,
        "covered_80": bool(q10 <= actual <= q90) if np.isfinite(actual) and np.isfinite(q10) else np.nan,
        "interval_width": q90 - q10 if np.isfinite(q10) and np.isfinite(q90) else np.nan,
        "impossible_price": point <= 0,
        "rationale": rationale,
    }

## 5. Run the numerical methods

For the no-news model-results agent, each point method also gets a small practice test: fit on the first 18 input months and predict the final six input months. This test ends at the cutoff, so it never uses the six future months being evaluated. The more complex reusable predictors do not receive a practice-test score because an 18-month fit is too short for a fair check.

The mean and median rows combine eligible numerical forecasts. They are derived combinations, not separately fitted models.

In [8]:
results_rows = []
diagnostics_rows = []
model_panels = {}

if RUN_NUMERICAL_METHODS:
    for scenario, cutoff in SCENARIOS.items():
        train = scenario_inputs[scenario]["value"]
        context = scenario_services[scenario].context(as_of=cutoff)
        assert len(context.get_series(COPPER_SERIES_ID)) == CONFIG.input_months
        panel_rows = []
        validation_paths = {}

        for model_name, (family, method) in POINT_METHODS.items():
            try:
                forecasts = point_method_forecast(train, max(CONFIG.horizons), method)
                validation_forecast = point_method_forecast(
                    train.iloc[: -CONFIG.validation_months],
                    CONFIG.validation_months,
                    method,
                )
                validation_paths[model_name] = validation_forecast
                validation_mae = float(
                    np.mean(np.abs(validation_forecast - train.iloc[-CONFIG.validation_months :].to_numpy()))
                )
                for horizon, point in zip(CONFIG.horizons, forecasts, strict=True):
                    results_rows.append(
                        result_row(
                            scenario=scenario,
                            cutoff=cutoff,
                            model=model_name,
                            family=family,
                            horizon=horizon,
                            point_forecast=float(point),
                        )
                    )
                    panel_rows.append(
                        {
                            "model": model_name,
                            "horizon_months": horizon,
                            "forecast": round(float(point), 2),
                            "past_validation_mae": round(validation_mae, 2),
                            "impossible_price": bool(point <= 0),
                        }
                    )
                diagnostics_rows.append(
                    {"scenario": scenario, "model": model_name, "status": "Completed", "detail": ""}
                )
            except (ValueError, np.linalg.LinAlgError) as exc:
                diagnostics_rows.append(
                    {"scenario": scenario, "model": model_name, "status": "Failed", "detail": str(exc)}
                )

        for model_name, factory in PROBABILISTIC_FACTORIES.items():
            try:
                np.random.seed(42)
                predictions = factory().predict(task, context)
                if len(predictions) != len(CONFIG.horizons):
                    raise ValueError(f"Expected {len(CONFIG.horizons)} forecasts, received {len(predictions)}.")
                impossible = False
                for horizon, prediction in zip(CONFIG.horizons, predictions, strict=True):
                    quantiles = {float(level): float(value) for level, value in prediction.payload.quantiles.items()}
                    point = float(prediction.payload.point_forecast)
                    impossible = impossible or point <= 0
                    results_rows.append(
                        result_row(
                            scenario=scenario,
                            cutoff=cutoff,
                            model=model_name,
                            family="Statistical model",
                            horizon=horizon,
                            point_forecast=point,
                            quantiles=quantiles,
                        )
                    )
                    panel_rows.append(
                        {
                            "model": model_name,
                            "horizon_months": horizon,
                            "forecast": round(point, 2),
                            "range_10_to_90": [round(quantiles[0.10], 2), round(quantiles[0.90], 2)],
                            "past_validation_mae": None,
                            "impossible_price": point <= 0,
                        }
                    )
                detail = "Contains a price at or below zero; retained for review." if impossible else ""
                status = "Warning" if impossible else "Completed"
                diagnostics_rows.append(
                    {"scenario": scenario, "model": model_name, "status": status, "detail": detail}
                )
            except Exception as exc:
                diagnostics_rows.append(
                    {"scenario": scenario, "model": model_name, "status": "Failed", "detail": str(exc)}
                )

        scenario_numerical = [row for row in results_rows if row["scenario"] == scenario]
        validation_actuals = train.iloc[-CONFIG.validation_months :].to_numpy()
        for ensemble_name, reducer in (("Mean ensemble", np.mean), ("Median ensemble", np.median)):
            validation_matrix = np.array(list(validation_paths.values()), dtype=float)
            validation_path = reducer(validation_matrix, axis=0)
            validation_mae = float(np.mean(np.abs(validation_path - validation_actuals)))
            for horizon in CONFIG.horizons:
                eligible = [
                    float(row["point_forecast"])
                    for row in scenario_numerical
                    if row["horizon"] == horizon
                    and np.isfinite(float(row["point_forecast"]))
                    and float(row["point_forecast"]) > 0
                ]
                if not eligible:
                    continue
                point = float(reducer(eligible))
                results_rows.append(
                    result_row(
                        scenario=scenario,
                        cutoff=cutoff,
                        model=ensemble_name,
                        family="Combined forecast",
                        horizon=horizon,
                        point_forecast=point,
                    )
                )
                panel_rows.append(
                    {
                        "model": ensemble_name,
                        "horizon_months": horizon,
                        "forecast": round(point, 2),
                        "past_validation_mae": round(validation_mae, 2),
                        "impossible_price": False,
                    }
                )
            diagnostics_rows.append(
                {"scenario": scenario, "model": ensemble_name, "status": "Completed", "detail": "Derived combination"}
            )

        model_panels[str(cutoff.date())] = panel_rows

numerical_results = pd.DataFrame(results_rows)
diagnostics = pd.DataFrame(diagnostics_rows)
print(f"Created {len(numerical_results):,} numerical forecast rows.")
display(diagnostics)

/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/statsmodels/t

Created 234 numerical forecast rows.


,scenario,model,status,detail
0,2011 commodity boom,Last value,Completed,
1,2011 commodity boom,Historical mean,Completed,
2,2011 commodity boom,Drift,Completed,
3,2011 commodity boom,Seasonal naive,Completed,
4,2011 commodity boom,12-month moving average,Completed,
5,2011 commodity boom,ETS trend,Completed,
6,2011 commodity boom,Damped Holt trend,Completed,
7,2011 commodity boom,AutoReg,Completed,
8,2011 commodity boom,Prophet,Completed,
9,2011 commodity boom,AutoARIMA,Completed,


## 6. Run the three agents

The three agent rows answer different questions:

- **History-only agent:** can a language model reason from the same 24 prices without outside information?
- **Model-results agent:** can a language model improve on the numerical methods by comparing their forecasts and their past-only practice-test errors, without news?
- **News agent:** does adding verified information published by the cutoff improve the forecast, and which global signals did it use?

All three run in the main comparison. They require configured model credentials and make multiple external calls. The model-results agent receives no target-period actual prices, errors, or news. The news agent must record the verified signals that affected its forecast in its rationale.

In [ ]:
agent_predictors = {}
if RUN_HISTORY_AGENT:
    agent_predictors["History-only agent"] = build_copper_agent_predictor(
        build_copper_basic_config(model=AGENT_MODEL)
    )
if RUN_MODEL_PANEL_AGENT:
    agent_predictors["Model-results agent (no news)"] = build_copper_model_panel_predictor(
        build_copper_model_panel_config(model=AGENT_MODEL),
        model_panels,
    )
if RUN_NEWS_AGENT:
    agent_predictors["News agent"] = build_copper_agent_predictor(
        build_copper_news_config(model=AGENT_MODEL)
    )

all_rows = list(results_rows)
for model_name, predictor in agent_predictors.items():
    for scenario, cutoff in SCENARIOS.items():
        try:
            predictions = predictor.predict(task, scenario_services[scenario].context(as_of=cutoff))
            if len(predictions) != len(CONFIG.horizons):
                raise ValueError(f"Expected {len(CONFIG.horizons)} forecasts, received {len(predictions)}.")
            for horizon, prediction in zip(CONFIG.horizons, predictions, strict=True):
                rationale_parts = [
                    str(prediction.metadata.get("rationale", "")),
                    str(prediction.metadata.get("horizon_rationale", "")),
                ]
                all_rows.append(
                    result_row(
                        scenario=scenario,
                        cutoff=cutoff,
                        model=model_name,
                        family="Agent",
                        horizon=horizon,
                        point_forecast=float(prediction.payload.point_forecast),
                        quantiles={
                            float(level): float(value) for level, value in prediction.payload.quantiles.items()
                        },
                        rationale=" ".join(part for part in rationale_parts if part).strip(),
                    )
                )
            diagnostics_rows.append(
                {"scenario": scenario, "model": model_name, "status": "Completed", "detail": ""}
            )
        except Exception as exc:
            diagnostics_rows.append(
                {"scenario": scenario, "model": model_name, "status": "Failed", "detail": str(exc)}
            )

results = pd.DataFrame(all_rows).sort_values(["scenario", "model", "horizon"]).reset_index(drop=True)
diagnostics = pd.DataFrame(diagnostics_rows)

for scenario, cutoff in SCENARIOS.items():
    assert len(scenario_services[scenario].context(as_of=cutoff).get_series(COPPER_SERIES_ID)) == 24
assert not results.duplicated(["scenario", "model", "horizon"]).any()
assert all(
    "actual" not in row and "error" not in row
    for panel in model_panels.values()
    for row in panel
)

if agent_predictors:
    display(results.loc[results["family"] == "Agent", ["scenario", "model", "horizon", "point_forecast", "status"]])
else:
    print("Agent runs are off. Enable an agent switch in the configuration cell to add its results.")

display(diagnostics.loc[diagnostics["status"] != "Completed"])
print(f"Result rows ready for analysis: {len(results):,}")

/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


## 7. Compare every forecast path with its two-year context

Each scenario gets its own chart. The charcoal line shows the exact 24 monthly prices supplied to every method. The black line after the cutoff shows the six target months where actual prices are available; pending 2026 months remain blank. Every successful method is drawn from the cutoff through its six predictions. Line style indicates the method family, and hovering identifies the method and value.

Only point forecasts are shown here so all methods remain comparable and the chart stays readable. Forecast-range calibration is evaluated separately in the scorecards.

In [ ]:
model_names = sorted(results["model"].unique())
model_palette = px.colors.qualitative.Dark24
model_colors = {
    model_name: model_palette[index % len(model_palette)]
    for index, model_name in enumerate(model_names)
}
family_dashes = {
    "Simple baseline": "dot",
    "Statistical model": "solid",
    "Combined forecast": "dashdot",
    "Agent": "dash",
}

scenario_figures = {}
for scenario, cutoff in SCENARIOS.items():
    context_frame = scenario_inputs[scenario]
    target_dates = pd.date_range(
        cutoff + pd.offsets.MonthBegin(1),
        periods=max(CONFIG.horizons),
        freq="MS",
    )
    target_actuals = copper.reindex(target_dates)["value"]
    scenario_results = results.loc[results["scenario"] == scenario].sort_values(
        ["family", "model", "horizon"]
    )

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=context_frame.index,
            y=context_frame["value"],
            mode="lines+markers",
            name="24-month context",
            line={"color": "#666666", "width": 3},
            marker={"size": 6},
            hovertemplate="%{x|%b %Y}<br>Context: $%{y:,.0f}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[cutoff, *target_dates.tolist()],
            y=[float(context_frame["value"].iloc[-1]), *target_actuals.tolist()],
            mode="lines+markers",
            name="6-month actuals",
            line={"color": "#111111", "width": 4},
            marker={"size": 8, "symbol": "circle"},
            connectgaps=False,
            hovertemplate="%{x|%b %Y}<br>Actual: $%{y:,.0f}<extra></extra>",
        )
    )

    hidden_nonpositive = []
    for model_name, grouped_frame in scenario_results.groupby("model", sort=True):
        model_frame = grouped_frame.sort_values("horizon")
        family = str(model_frame["family"].iloc[0])
        has_nonpositive = bool((model_frame["point_forecast"] <= 0).any())
        if has_nonpositive:
            hidden_nonpositive.append(model_name)
        figure.add_trace(
            go.Scatter(
                x=[cutoff, *model_frame["forecast_date"].tolist()],
                y=[float(context_frame["value"].iloc[-1]), *model_frame["point_forecast"].tolist()],
                mode="lines+markers",
                name=model_name,
                legendgroup=family,
                legendgrouptitle_text=family,
                visible="legendonly" if has_nonpositive else True,
                line={
                    "color": model_colors[model_name],
                    "width": 3 if family == "Agent" else 2,
                    "dash": family_dashes[family],
                },
                marker={"size": 6},
                opacity=0.9,
                customdata=np.repeat(family, len(model_frame) + 1),
                hovertemplate=(
                    "%{x|%b %Y}<br>%{fullData.name}: $%{y:,.0f}"
                    "<br>%{customdata}<extra></extra>"
                ),
            )
        )

    figure.add_vrect(
        x0=cutoff,
        x1=target_dates.max(),
        fillcolor="#dcebea",
        opacity=0.28,
        line_width=0,
        layer="below",
    )
    figure.add_vline(x=cutoff, line_color="#111111", line_width=2, line_dash="dash")
    figure.add_annotation(
        x=cutoff,
        y=1.03,
        xref="x",
        yref="paper",
        text="Forecast cutoff",
        showarrow=False,
        xanchor="left",
        font={"size": 12, "color": "#333333"},
    )
    if hidden_nonpositive:
        figure.add_annotation(
            x=0,
            y=-0.18,
            xref="paper",
            yref="paper",
            text=(
                "Hidden by default because the path contains non-positive prices: "
                + ", ".join(hidden_nonpositive)
                + ". Select the legend entry to inspect it."
            ),
            showarrow=False,
            xanchor="left",
            align="left",
            font={"size": 11, "color": "#8a2f2f"},
        )

    figure.update_layout(
        title=f"{scenario}: 24-month context and six-month forecasts",
        template="plotly_white",
        height=650,
        margin={"l": 75, "r": 300, "t": 90, "b": 115},
        hovermode="x unified",
        legend={
            "x": 1.02,
            "y": 1,
            "xanchor": "left",
            "yanchor": "top",
            "groupclick": "toggleitem",
            "tracegroupgap": 8,
        },
        xaxis_title="Month",
        yaxis_title="Copper price (USD per metric ton)",
    )
    figure.update_xaxes(
        range=[context_frame.index.min() - pd.DateOffset(days=12), target_dates.max() + pd.DateOffset(days=12)]
    )
    scenario_figures[scenario] = figure
    figure.show()

## 8. Quantify forecast accuracy

Only rows marked **Scored** enter these tables. Pending 2026 months remain visible in the forecast table but cannot affect rankings. MASE compares each forecast with the last-value error from the same scenario, avoiding an unfair comparison across very different copper-price levels.

In [ ]:
scored = results.loc[results["status"] == "Scored"].copy()
last_value_mae = (
    scored.loc[scored["model"] == "Last value"].groupby("scenario")["absolute_error"].mean()
)
scored["scaled_absolute_error"] = scored.apply(
    lambda row: row["absolute_error"] / last_value_mae[row["scenario"]]
    if last_value_mae[row["scenario"]] > 0
    else np.nan,
    axis=1,
)


def summarize_scores(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    """Create point and forecast-range metrics for the requested groups."""
    rows = []
    for group_key, group in frame.groupby(group_columns, dropna=False):
        key_values = group_key if isinstance(group_key, tuple) else (group_key,)
        row = dict(zip(group_columns, key_values, strict=True))
        has_full_ranges = group["crps"].notna().any()
        has_80_ranges = group["covered_80"].notna().any()
        row.update(
            {
                "MAE": group["absolute_error"].mean(),
                "RMSE": np.sqrt(group["squared_error"].mean()),
                "sMAPE (%)": group["absolute_percentage_component"].mean(),
                "MASE": group["scaled_absolute_error"].mean(),
                "Bias": group["error"].mean(),
                "Direction accuracy": group["direction_correct"].astype(float).mean(),
                "CRPS": group["crps"].mean() if has_full_ranges else np.nan,
                "80% coverage": group.loc[group["covered_80"].notna(), "covered_80"].astype(float).mean()
                if has_80_ranges
                else np.nan,
                "Average interval width": group["interval_width"].mean() if has_80_ranges else np.nan,
                "Scored forecasts": len(group),
                "Impossible forecasts": int(group["impossible_price"].sum()),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


overall_scorecard = summarize_scores(scored, ["model", "family"]).sort_values(["MAE", "RMSE"])
scenario_scorecard = summarize_scores(scored, ["scenario", "model", "family"])
scenario_scorecard["Scenario rank by MAE"] = scenario_scorecard.groupby("scenario")["MAE"].rank(method="min")
scenario_scorecard = scenario_scorecard.sort_values(["scenario", "Scenario rank by MAE"])

metric_columns = [
    "model",
    "family",
    "MAE",
    "RMSE",
    "sMAPE (%)",
    "MASE",
    "Bias",
    "Direction accuracy",
    "CRPS",
    "80% coverage",
    "Average interval width",
    "Scored forecasts",
    "Impossible forecasts",
]
display(overall_scorecard[metric_columns].round(3))

,model,family,MAE,RMSE,sMAPE (%),MASE,Bias,Direction accuracy,CRPS,80% coverage,Average interval width,Scored forecasts,Impossible forecasts
11,Median ensemble,Combined forecast,711.101,811.408,8.901,0.973,-191.218,0.500,NaN,NaN,NaN,16,0
7,History-only agent,Agent,721.051,854.968,9.616,0.979,-269.934,0.500,496.386,0.688,1650.000,16,0
9,Last value,Simple baseline,739.192,855.427,9.207,1.000,-267.043,0.000,NaN,NaN,NaN,16,0
10,Mean ensemble,Combined forecast,791.281,984.330,10.050,1.018,-665.173,0.562,NaN,NaN,NaN,16,0
13,News agent,Agent,809.548,912.230,10.357,1.128,-169.934,0.375,643.321,0.375,1143.750,16,0
2,AutoReg,Statistical model,818.729,989.188,8.737,1.101,450.017,0.562,NaN,NaN,NaN,16,0
12,Model-results agent (no news),Agent,886.051,1004.289,10.968,1.238,-147.434,0.312,622.263,0.438,1719.375,16,0
14,Prophet,Statistical model,1042.291,1176.515,12.047,1.362,447.064,0.812,985.530,0.000,256.205,16,0
3,Damped Holt trend,Statistical model,1061.456,1255.966,13.224,1.533,38.774,0.312,NaN,NaN,NaN,16,0
1,AutoARIMA,Statistical model,1082.048,1260.646,12.411,1.569,102.432,0.500,780.802,0.250,1590.532,16,0


## 9. Examine where errors occur

No single average explains everything. The next views show overall error, how error changes with forecast distance, whether forecasts tend to be high or low, and which methods lead within each period.

In [ ]:
mae_chart_data = overall_scorecard.sort_values("MAE", ascending=True)
mae_figure = px.bar(
    mae_chart_data,
    x="MAE",
    y="model",
    color="family",
    orientation="h",
    title="Average forecast miss across observed target months",
    labels={"MAE": "MAE (USD per metric ton)", "model": "Method", "family": "Method family"},
    color_discrete_map={
        "Simple baseline": "#777777",
        "Statistical model": TEAL,
        "Combined forecast": COPPER,
        "Agent": "#3f7d44",
    },
    text_auto=".0f",
)
mae_figure.update_layout(template="plotly_white", height=560, legend={"orientation": "h", "y": 1.10})
mae_figure.show()

horizon_errors = scored.pivot_table(index="model", columns="horizon", values="absolute_error", aggfunc="mean")
horizon_errors = horizon_errors.loc[overall_scorecard.sort_values("MAE")["model"]]
heatmap = px.imshow(
    horizon_errors,
    aspect="auto",
    color_continuous_scale=[[0, "#f7f5f2"], [0.5, "#c98862"], [1, "#3a2922"]],
    title="Average absolute error by forecast distance",
    labels={"x": "Months after cutoff", "y": "Method", "color": "Absolute error"},
    text_auto=".0f",
)
heatmap.update_layout(template="plotly_white", height=620)
heatmap.show()

In [ ]:
signed_error_figure = px.strip(
    scored,
    x="model",
    y="error",
    color="scenario",
    title="Signed forecast errors: above zero means too high",
    labels={"model": "Method", "error": "Forecast minus actual (USD per metric ton)", "scenario": "Period"},
    color_discrete_sequence=[COPPER, TEAL, "#315b7d"],
)
signed_error_figure.add_hline(y=0, line_color=CHARCOAL, line_width=2)
signed_error_figure.update_layout(template="plotly_white", height=520, xaxis_tickangle=-40)
signed_error_figure.show()

display(
    scenario_scorecard.loc[
        :,
        ["scenario", "Scenario rank by MAE", "model", "MAE", "RMSE", "MASE", "Bias", "Scored forecasts"],
    ].round(3)
)

,scenario,Scenario rank by MAE,model,MAE,RMSE,MASE,Bias,Scored forecasts
10,2011 commodity boom,1.0,Mean ensemble,333.059,435.508,0.547,-260.383,6
14,2011 commodity boom,2.0,Prophet,586.388,640.676,0.963,-137.220,6
7,2011 commodity boom,3.0,History-only agent,594.485,670.820,0.977,594.485,6
9,2011 commodity boom,4.0,Last value,608.756,670.531,1.000,608.756,6
11,2011 commodity boom,5.0,Median ensemble,646.204,698.699,1.062,646.204,6
2,2011 commodity boom,6.0,AutoReg,685.923,748.181,1.127,685.923,6
13,2011 commodity boom,7.0,News agent,852.819,918.917,1.401,852.819,6
12,2011 commodity boom,8.0,Model-results agent (no news),969.485,1043.043,1.593,969.485,6
0,2011 commodity boom,9.0,12-month moving average,1302.229,1332.226,2.139,-1302.229,6
3,2011 commodity boom,10.0,Damped Holt trend,1457.474,1564.765,2.394,1457.474,6


## 10. Main comparison: three agents versus the best benchmark

This is the notebook's primary comparison. It selects the lowest-MAE **single statistical model or simple baseline** across the scored months, then compares that benchmark with the history-only, model-results, and news agents on identical observations. Mean and median forecast combinations are excluded from benchmark selection because they are not single statistical or naive methods.

The news-signal audit reports only signals the agent says affected its forecast. These rationale-based labels make the forecast inspectable, but they are not causal feature importance. The 2011 and 2020 agent results also need caution because a modern language model may have encountered those historical periods during training; the observed 2026 months provide the cleaner comparison.

In [ ]:
AGENT_NAMES = [
    "History-only agent",
    "Model-results agent (no news)",
    "News agent",
]
BENCHMARK_FAMILIES = ["Simple baseline", "Statistical model"]

benchmark_candidates = overall_scorecard.loc[
    overall_scorecard["family"].isin(BENCHMARK_FAMILIES)
    & (overall_scorecard["Impossible forecasts"] == 0)
].sort_values(["MAE", "RMSE"])
if benchmark_candidates.empty:
    raise ValueError("No valid statistical or simple benchmark is available.")

benchmark_name = str(benchmark_candidates.iloc[0]["model"])
comparison_names = [benchmark_name, *AGENT_NAMES]
main_comparison = overall_scorecard.loc[
    overall_scorecard["model"].isin(comparison_names)
].copy()

benchmark_errors = scored.loc[
    scored["model"] == benchmark_name,
    ["scenario", "horizon", "absolute_error"],
].rename(columns={"absolute_error": "benchmark_absolute_error"})
paired_agent_errors = scored.loc[
    scored["model"].isin(AGENT_NAMES),
    ["scenario", "horizon", "model", "absolute_error"],
].merge(benchmark_errors, on=["scenario", "horizon"], how="inner", validate="many_to_one")
paired_agent_errors["beats_benchmark"] = (
    paired_agent_errors["absolute_error"] < paired_agent_errors["benchmark_absolute_error"]
)

if paired_agent_errors.empty:
    paired_summary = pd.DataFrame(
        columns=[
            "model",
            "Paired forecasts",
            "Agent MAE on paired rows",
            "Benchmark MAE on paired rows",
            "Win rate vs benchmark",
            "MAE improvement vs benchmark (%)",
        ]
    )
else:
    paired_summary = paired_agent_errors.groupby("model", as_index=False).agg(
        **{
            "Paired forecasts": ("absolute_error", "size"),
            "Agent MAE on paired rows": ("absolute_error", "mean"),
            "Benchmark MAE on paired rows": ("benchmark_absolute_error", "mean"),
            "Win rate vs benchmark": ("beats_benchmark", "mean"),
        }
    )
    paired_summary["MAE improvement vs benchmark (%)"] = 100 * (
        paired_summary["Benchmark MAE on paired rows"] - paired_summary["Agent MAE on paired rows"]
    ) / paired_summary["Benchmark MAE on paired rows"]

main_comparison = main_comparison.merge(paired_summary, on="model", how="left")
benchmark_mask = main_comparison["model"] == benchmark_name
main_comparison.loc[benchmark_mask, "Paired forecasts"] = main_comparison.loc[
    benchmark_mask, "Scored forecasts"
]
main_comparison.loc[benchmark_mask, "Agent MAE on paired rows"] = main_comparison.loc[
    benchmark_mask, "MAE"
]
main_comparison.loc[benchmark_mask, "Benchmark MAE on paired rows"] = main_comparison.loc[
    benchmark_mask, "MAE"
]
main_comparison.loc[benchmark_mask, "MAE improvement vs benchmark (%)"] = 0.0
main_comparison["Rank by MAE"] = main_comparison["MAE"].rank(method="min").astype(int)
main_comparison = main_comparison.sort_values(["Rank by MAE", "RMSE"])

missing_agents = [name for name in AGENT_NAMES if name not in set(main_comparison["model"])]
print(f"Selected benchmark: {benchmark_name} (lowest MAE among single statistical and simple methods).")
if missing_agents:
    print("Missing agent results:", ", ".join(missing_agents))
    display(
        diagnostics.loc[
            (diagnostics["model"].isin(missing_agents)) & (diagnostics["status"] != "Completed")
        ]
    )

display(
    main_comparison[
        [
            "Rank by MAE",
            "model",
            "family",
            "MAE",
            "RMSE",
            "MASE",
            "Bias",
            "Direction accuracy",
            "CRPS",
            "80% coverage",
            "MAE improvement vs benchmark (%)",
            "Win rate vs benchmark",
            "Paired forecasts",
        ]
    ].round(3)
)

comparison_figure = px.bar(
    main_comparison,
    x="model",
    y="MAE",
    color="family",
    title=f"Main comparison: three agent designs versus {benchmark_name}",
    labels={"model": "Method", "MAE": "MAE (USD per metric ton)", "family": "Method family"},
    color_discrete_map={"Agent": COPPER, "Simple baseline": CHARCOAL, "Statistical model": TEAL},
)
comparison_figure.update_layout(template="plotly_white", height=470, xaxis_tickangle=-20)
comparison_figure.show()

scenario_main = scenario_scorecard.loc[
    scenario_scorecard["model"].isin(comparison_names)
].copy()
scenario_main["Rank among compared methods"] = scenario_main.groupby("scenario")["MAE"].rank(method="min")
display(
    scenario_main[
        [
            "scenario",
            "Rank among compared methods",
            "model",
            "MAE",
            "RMSE",
            "Bias",
            "Direction accuracy",
            "Scored forecasts",
        ]
    ].round(3)
)

SIGNAL_KEYWORDS = {
    "Mine supply and disruptions": ("mine", "mining", "smelter", "treatment charge", "supply disruption"),
    "Exchange inventories": ("inventory", "inventories", "warehouse", "lme", "comex", "shfe"),
    "China demand": ("china", "chinese", "property sector", "manufacturing demand"),
    "US dollar and interest rates": ("us dollar", "u.s. dollar", "dxy", "federal reserve", "interest rate"),
    "Energy-transition demand": ("energy transition", "electric vehicle", "renewable", "power grid"),
    "Trade and geopolitics": ("tariff", "trade tension", "geopolit", "sanction", "conflict"),
    "Published analyst outlooks": ("analyst outlook", "price target", "published outlook"),
}


def identify_global_signals(rationale: str) -> list[str]:
    """Map the news agent's written rationale to broad signal categories."""
    normalized = rationale.lower()
    return [
        category
        for category, keywords in SIGNAL_KEYWORDS.items()
        if any(keyword in normalized for keyword in keywords)
    ]


news_rows = results.loc[
    (results["model"] == "News agent") & results["rationale"].str.strip().ne(""),
    ["scenario", "horizon", "rationale"],
].copy()
if news_rows.empty:
    print("No news-agent rationale is available for signal analysis.")
else:
    news_signal_review = news_rows.groupby("scenario", as_index=False).agg(
        **{
            "Forecast horizons": ("horizon", "nunique"),
            "News-agent rationale": (
                "rationale",
                lambda values: "\n\n".join(dict.fromkeys(value for value in values if value)),
            ),
        }
    )
    news_signal_review["Global signals cited"] = news_signal_review["News-agent rationale"].map(
        lambda text: ", ".join(identify_global_signals(text)) or "No named global signal"
    )
    display(
        news_signal_review[
            ["scenario", "Forecast horizons", "Global signals cited", "News-agent rationale"]
        ]
    )

    signal_rows = [
        {"scenario": scenario, "signal": signal}
        for scenario, rationale in zip(
            news_signal_review["scenario"],
            news_signal_review["News-agent rationale"],
            strict=True,
        )
        for signal in identify_global_signals(rationale)
    ]
    if signal_rows:
        signal_coverage = (
            pd.DataFrame(signal_rows)
            .groupby("signal", as_index=False)
            .agg(**{"Periods cited": ("scenario", "nunique")})
            .sort_values(["Periods cited", "signal"], ascending=[False, True])
        )
        display(signal_coverage)

if not missing_agents:
    best_agent = main_comparison.loc[main_comparison["family"] == "Agent"].sort_values("MAE").iloc[0]
    news_score = main_comparison.loc[main_comparison["model"] == "News agent"].iloc[0]
    cited_signals = sorted(
        {
            signal
            for rationale in news_rows["rationale"]
            for signal in identify_global_signals(rationale)
        }
    )
    print(
        f"Best agent by MAE: {best_agent['model']} at {best_agent['MAE']:,.0f} USD per metric ton."
    )
    print(
        f"News agent MAE: {news_score['MAE']:,.0f}; paired improvement versus {benchmark_name}: "
        f"{news_score['MAE improvement vs benchmark (%)']:.1f}%."
    )
    print("Global signal categories cited by the news agent:", ", ".join(cited_signals) or "none")

Selected benchmark: Last value (lowest MAE among single statistical and simple methods).


,Rank by MAE,model,family,MAE,RMSE,MASE,Bias,Direction accuracy,CRPS,80% coverage,MAE improvement vs benchmark (%),Win rate vs benchmark,Paired forecasts
0,1,History-only agent,Agent,721.051,854.968,0.979,-269.934,0.500,496.386,0.688,2.454,0.500,16.0
1,2,Last value,Simple baseline,739.192,855.427,1.000,-267.043,0.000,NaN,NaN,0.000,NaN,16.0
2,3,News agent,Agent,809.548,912.230,1.128,-169.934,0.375,643.321,0.375,-9.518,0.375,16.0
3,4,Model-results agent (no news),Agent,886.051,1004.289,1.238,-147.434,0.312,622.263,0.438,-19.867,0.312,16.0


,scenario,Rank among compared methods,model,MAE,RMSE,Bias,Direction accuracy,Scored forecasts
7,2011 commodity boom,1.0,History-only agent,594.485,670.820,594.485,0.500,6
9,2011 commodity boom,2.0,Last value,608.756,670.531,608.756,0.000,6
13,2011 commodity boom,3.0,News agent,852.819,918.917,852.819,0.000,6
12,2011 commodity boom,4.0,Model-results agent (no news),969.485,1043.043,969.485,0.000,6
25,2020 pandemic shock,1.0,Last value,798.613,986.288,-757.059,0.000,6
28,2020 pandemic shock,2.0,Model-results agent (no news),877.034,1076.902,-846.358,0.167,6
29,2020 pandemic shock,3.0,News agent,889.691,1038.997,-889.691,0.333,6
23,2020 pandemic shock,4.0,History-only agent,928.701,1107.372,-914.691,0.167,6
39,2026 recent period,1.0,History-only agent,599.426,639.902,-599.426,1.000,4
45,2026 recent period,2.0,News agent,624.426,665.408,-624.426,1.000,4


,scenario,Forecast horizons,Global signals cited,News-agent rationale
0,2011 commodity boom,6,"Mine supply and disruptions, China demand, US ...",Global signals used: (1) Anticipated 2011 glob...
1,2020 pandemic shock,6,"Mine supply and disruptions, Exchange inventor...",Global signals used: COVID-19 pandemic shock (...
2,2026 recent period,6,"Mine supply and disruptions, Trade and geopoli...",Global signals used: Structural supply-side te...


,signal,Periods cited
2,Mine supply and disruptions,3
0,China demand,2
1,Exchange inventories,1
3,Trade and geopolitics,1
4,US dollar and interest rates,1


Best agent by MAE: History-only agent at 721 USD per metric ton.
News agent MAE: 810; paired improvement versus Last value: -9.5%.
Global signal categories cited by the news agent: China demand, Exchange inventories, Mine supply and disruptions, Trade and geopolitics, US dollar and interest rates


In [ ]:
if "news_signal_review" in globals():
    for scenario, rationale in zip(
        news_signal_review["scenario"],
        news_signal_review["News-agent rationale"],
        strict=True,
    ):
        print(f"\n{scenario} — complete news-agent rationale:\n{rationale}")


2011 commodity boom — complete news-agent rationale:
Global signals used: (1) Anticipated 2011 global copper market deficit (~400k tonnes) due to supply constraints, expected upward effect, source: industry consensus as of Feb 2011; (2) Rapid Chinese industrialization and demand, expected upward effect, source: market reporting as of Feb 2011; (3) US Dollar volatility as safe-haven vs commodity inverse relationship, potential downward risk, source: market reporting as of Feb 2011. The forecast reflects the strong momentum entering Feb 2011, balanced by the historical difficulty of sustaining record-breaking price growth over extended periods. Prices are at a record high. The immediate outlook remains bullish due to supply constraints and strong Chinese demand, sustaining momentum into the next month.

Global signals used: (1) Anticipated 2011 global copper market deficit (~400k tonnes) due to supply constraints, expected upward effect, source: industry consensus as of Feb 2011; (2) Ra

### What the saved run shows

- **Last value is the benchmark to beat:** MAE was **$739 per metric ton** across the 16 observed targets.
- **History-only came closest:** MAE was **$759**, 2.6% worse than the benchmark, while beating it on 43.8% of paired months.
- **Model-results without news did not add value:** MAE was **$859**, 16.1% worse than the benchmark, with a 31.2% monthly win rate.
- **News performed worst overall:** MAE was **$913**, 23.5% worse than the benchmark, with a 12.5% monthly win rate. Its average forecast was $373 below the actual price.
- **The recent period is more informative:** for the four observed 2026 targets, history-only and model-results beat last value, but news was last with MAE of **$1,099** and the same $1,099 downward bias.
- **Signals incorporated:** the news rationale cited mine supply or disruptions, exchange inventories, and Chinese demand in all three periods; the US dollar or interest rates appeared in one. The detailed rationales also identify Chinese industrial demand in 2011, the pandemic demand shock in 2020, and an International Copper Study Group supply-growth projection in 2026.

The news agent used economically relevant evidence, but that evidence pushed its forecasts too low in this small sample. This is descriptive rather than definitive: there are only 16 observed targets, two 2026 months are still pending, and the historical agent runs carry language-model training-data risk.

## 11. Secondary all-method leaderboard

For completeness, this table includes every observed month and every method. The primary decision table is Section 10: the three agent approaches versus the best single statistical or naive benchmark.

Agent rows from 2011 and 2020 may reflect historical information encountered during language-model training, even though live news search is restricted to the forecast cutoff. Results for 2026 remain incomplete until August and September prices are available.

In [ ]:
combined_leaderboard = overall_scorecard.copy()
combined_leaderboard["Agent result note"] = np.where(
    combined_leaderboard["family"] == "Agent",
    "Includes 2011 and 2020, which may reflect historical information learned during model training.",
    "",
)
display(combined_leaderboard[metric_columns + ["Agent result note"]].round(3))

best_point = combined_leaderboard.iloc[0]
range_methods = combined_leaderboard.dropna(subset=["CRPS"]).sort_values("CRPS")
print(
    f"Lowest combined MAE: {best_point['model']} at {best_point['MAE']:,.0f} USD per metric ton."
)
if not range_methods.empty:
    best_range = range_methods.iloc[0]
    print(
        f"Lowest CRPS among methods with full forecast ranges: {best_range['model']} "
        f"at {best_range['CRPS']:,.0f}."
    )
print(
    "The period-level table should remain the main check: leaders change across periods, "
    "and the 2026 ranking currently uses only four observed months."
)
if int(scored["impossible_price"].sum()) > 0:
    impossible_models = sorted(scored.loc[scored["impossible_price"], "model"].unique())
    print("Methods with at least one impossible price forecast:", ", ".join(impossible_models))

,model,family,MAE,RMSE,sMAPE (%),MASE,Bias,Direction accuracy,CRPS,80% coverage,Average interval width,Scored forecasts,Impossible forecasts,Agent result note
11,Median ensemble,Combined forecast,711.101,811.408,8.901,0.973,-191.218,0.500,NaN,NaN,NaN,16,0,
7,History-only agent,Agent,721.051,854.968,9.616,0.979,-269.934,0.500,496.386,0.688,1650.000,16,0,"Includes 2011 and 2020, which may reflect hist..."
9,Last value,Simple baseline,739.192,855.427,9.207,1.000,-267.043,0.000,NaN,NaN,NaN,16,0,
10,Mean ensemble,Combined forecast,791.281,984.330,10.050,1.018,-665.173,0.562,NaN,NaN,NaN,16,0,
13,News agent,Agent,809.548,912.230,10.357,1.128,-169.934,0.375,643.321,0.375,1143.750,16,0,"Includes 2011 and 2020, which may reflect hist..."
2,AutoReg,Statistical model,818.729,989.188,8.737,1.101,450.017,0.562,NaN,NaN,NaN,16,0,
12,Model-results agent (no news),Agent,886.051,1004.289,10.968,1.238,-147.434,0.312,622.263,0.438,1719.375,16,0,"Includes 2011 and 2020, which may reflect hist..."
14,Prophet,Statistical model,1042.291,1176.515,12.047,1.362,447.064,0.812,985.530,0.000,256.205,16,0,
3,Damped Holt trend,Statistical model,1061.456,1255.966,13.224,1.533,38.774,0.312,NaN,NaN,NaN,16,0,
1,AutoARIMA,Statistical model,1082.048,1260.646,12.411,1.569,102.432,0.500,780.802,0.250,1590.532,16,0,


Lowest combined MAE: Median ensemble at 711 USD per metric ton.
Lowest CRPS among methods with full forecast ranges: History-only agent at 496.
The period-level table should remain the main check: leaders change across periods, and the 2026 ranking currently uses only four observed months.
Methods with at least one impossible price forecast: Kalman


## 12. Validate and optionally export the results

The assertions below protect the main experiment rules: 24 input observations, six forecast positions per method and period, no duplicated results, no invented ranges for point-only methods, and pending targets kept out of scoring.

Exports are off by default. When enabled, they write the forecast-level results, combined scorecard, diagnostics, and configuration under `data/predictions/copper_monthly_backtest/`.

In [ ]:
for scenario, cutoff in SCENARIOS.items():
    visible = scenario_services[scenario].context(as_of=cutoff).get_series(COPPER_SERIES_ID)
    assert len(visible) == CONFIG.input_months
    assert pd.Timestamp(visible["timestamp"].max()) == cutoff

assert not results.duplicated(["scenario", "model", "horizon"]).any()
assert results.groupby(["scenario", "model"])["horizon"].nunique().eq(len(CONFIG.horizons)).all()
assert results.loc[results["family"].isin(["Simple baseline", "Combined forecast"]), ["q10", "q90"]].isna().all().all()
assert results.loc[results["status"] == "Scored", "actual"].notna().all()
assert results.loc[results["status"] == "Pending", "actual"].isna().all()
assert len(results.loc[(results["scenario"] == "2026 recent period") & (results["status"] == "Pending")]) == 2 * results.loc[
    results["scenario"] == "2026 recent period", "model"
].nunique()

print("Validation passed: inputs, forecast counts, ranges, and pending targets follow the experiment rules.")

if EXPORT_RESULTS:
    export_dir = ROOT / "data" / "predictions" / "copper_monthly_backtest"
    export_dir.mkdir(parents=True, exist_ok=True)
    export_frame = results.copy()
    export_frame["quantiles"] = export_frame["quantiles"].map(
        lambda value: json.dumps(value, sort_keys=True) if isinstance(value, dict) else ""
    )
    export_frame.to_csv(export_dir / "forecasts.csv", index=False)
    combined_leaderboard.to_csv(export_dir / "scorecard.csv", index=False)
    diagnostics.to_csv(export_dir / "diagnostics.csv", index=False)
    config_payload = {
        **asdict(CONFIG),
        "cache_path": str(CONFIG.cache_path),
        "scenarios": {name: str(cutoff.date()) for name, cutoff in SCENARIOS.items()},
        "agent_model": AGENT_MODEL,
        "run_history_agent": RUN_HISTORY_AGENT,
        "run_model_panel_agent": RUN_MODEL_PANEL_AGENT,
        "run_news_agent": RUN_NEWS_AGENT,
        "created_at_utc": datetime.now(tz=timezone.utc).isoformat(),
    }
    (export_dir / "configuration.json").write_text(json.dumps(config_payload, indent=2), encoding="utf-8")
    print("Exported results to", export_dir)
else:
    print("Set EXPORT_RESULTS=True to write result files.")

Validation passed: inputs, forecast counts, ranges, and pending targets follow the experiment rules.
Set EXPORT_RESULTS=True to write result files.
